# 02 · Preprocesamiento: VIF y SMOTE

1. Detectar multicolinealidad con el **Factor de Inflación de la Varianza (VIF)** y descartar variables con VIF > 5.
2. Dividir train/test **antes** de balancear.
3. Aplicar **SMOTE solo sobre el conjunto de entrenamiento**.

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
FEATURES = [
    "promedio", "inasistencias", "conducta", "horas_estudio", "nivel_edu_padres",
    "acceso_tecnologia", "tiempo_disponible_padres", "comunicacion_familiar",
    "cursos_desaprobados", "tareas_entregadas_pct",
]
TARGET = "en_riesgo"

df = pd.read_csv("../data/dataset.csv")

In [ ]:
X = add_constant(df[FEATURES])
vif = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
}).query("variable != 'const'").sort_values("VIF", ascending=False)
vif

In [ ]:
UMBRAL_VIF = 5
seleccionadas = vif.loc[vif["VIF"] <= UMBRAL_VIF, "variable"].tolist()
print("Variables seleccionadas:", seleccionadas)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    df[seleccionadas], df[TARGET], test_size=0.2, stratify=df[TARGET], random_state=42
)
X_bal, y_bal = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
print("Antes:", y_tr.value_counts().to_dict(), "→ Después:", y_bal.value_counts().to_dict())

In [ ]:
pd.concat([X_bal, y_bal], axis=1).to_csv("../data/train_balanceado.csv", index=False)
pd.concat([X_te, y_te], axis=1).to_csv("../data/test.csv", index=False)